# YouTube Music Playlist → Crossfade Mix

Downloads audio from a YouTube Music playlist and merges all tracks into one MP3 with seamless crossfades.

In [ ]:
!pip install yt-dlp pydub -q

## Configuration

Set your playlist URL and preferences below.

In [ ]:
# ─── Configuration ───────────────────────────────────────────
PLAYLIST_URL = "https://music.youtube.com/playlist?list=YOUR_PLAYLIST_ID"
OUTPUT_DIR = "downloaded_songs"
OUTPUT_FILE = "crossfade_mix.mp3"
CROSSFADE_MS = 5000           # duration of crossfade in milliseconds
AUDIO_FORMAT = "mp3"          # mp3, m4a, opus, etc.
AUDIO_QUALITY = 0             # 0=best, 9=worst (for mp3)
KEEP_DOWNLOADS = False        # delete individual songs after mixing?
# ────────────────────────────────────────────────────────────

## Step 1: Download Playlist

In [ ]:
import os
import glob
import subprocess
import sys
import shutil
from pathlib import Path

try:
    from pydub import AudioSegment
    from pydub.exceptions import CouldntDecodeError
except ImportError:
    print("pydub not installed. Run the first cell first.")
    sys.exit(1)


def check_deps():
    """Verify that required external tools are available."""
    for cmd in ["yt-dlp", "ffmpeg"]:
        if not shutil.which(cmd):
            print(f"ERROR: '{cmd}' not found on PATH. Install it and try again.")
            sys.exit(1)
    print("✓ Dependencies found: yt-dlp, ffmpeg")


def download_playlist(url, out_dir, audio_format, quality):
    """Download each track in a YouTube playlist as an audio file."""
    os.makedirs(out_dir, exist_ok=True)

    cmd = [
        "yt-dlp",
        "-x",                          # extract audio
        "--audio-format", audio_format,
        "--audio-quality", str(quality),
        "--embed-thumbnail",
        "--add-metadata",
        "--output", f"{out_dir}/%(title)s.%(ext)s",
        "--no-playlist-reverse",
        "--yes-playlist",
        url,
    ]

    print(f"Downloading playlist: {url}")
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print("yt-dlp failed:")
        print(result.stderr)
        sys.exit(1)

    print("✓ Download complete")


check_deps()
download_playlist(PLAYLIST_URL, OUTPUT_DIR, AUDIO_FORMAT, AUDIO_QUALITY)

## Step 2: Crossfade & Export

In [ ]:
def get_audio_files(directory, extension):
    """Return sorted list of audio files with the given extension."""
    pattern = os.path.join(directory, f"*.{extension}")
    return sorted(glob.glob(pattern))


def build_crossfade_mix(file_list, crossfade_ms, output_path):
    """Concatenate audio files with crossfade and export as MP3."""
    if not file_list:
        print("No audio files found. Did the download produce any files?")
        return

    print(f"Concatenating {len(file_list)} tracks with {crossfade_ms}ms crossfade...")

    combined = None
    loaded_count = 0

    for i, fpath in enumerate(file_list, 1):
        try:
            seg = AudioSegment.from_file(fpath)
        except (CouldntDecodeError, Exception) as e:
            print(f"  [{i}/{len(file_list)}] SKIP (corrupt/unreadable): {Path(fpath).name} — {e}")
            continue

        if combined is None:
            combined = seg
        else:
            combined = combined.append(seg, crossfade=crossfade_ms)

        loaded_count += 1
        print(f"  [{i}/{len(file_list)}] ✓ {Path(fpath).name}  ({len(seg) / 1000:.1f}s)")

    if combined is None:
        print("No valid audio files were loaded. Nothing to export.")
        return

    print(f"\nExporting {loaded_count} tracks → {output_path}  ({len(combined) / 1000:.1f}s total)")
    combined.export(output_path, format="mp3", bitrate="320k")
    print("✓ Done!")


files = get_audio_files(OUTPUT_DIR, AUDIO_FORMAT)
build_crossfade_mix(files, CROSSFADE_MS, OUTPUT_FILE)

## Step 3: (Optional) Clean Up

In [ ]:
if KEEP_DOWNLOADS:
    print(f"Individual downloads kept in '{OUTPUT_DIR}/'")
else:
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    print(f"Removed '{OUTPUT_DIR}/' — only {OUTPUT_FILE} remains")